# Week 1 Assignment — Time Series Foundations
### Multi-Agent Forecasting Project

**Name:**  
**Date:**  

---

This assignment covers the fundamentals of time series analysis that will form the backbone of our multi-agent forecasting system. Complete all sections. Cells marked `# YOUR CODE HERE` require your implementation.

**Topics covered:**
1. Pandas Time Series Basics
2. Visualization with Matplotlib
3. Time Series Components (Trend, Seasonality, Cyclic, Random)
4. Moving Averages
5. Lag Features
6. Putting It All Together

In [ ]:
# Run this cell first — installs/imports everything you need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.seasonal import seasonal_decompose

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("All libraries imported successfully!")

---
## Section 1 — Pandas Time Series Basics

Pandas has first-class support for time series data. The key building blocks are:
- `pd.to_datetime()` — parse strings/integers into datetime objects
- `pd.date_range()` — generate a sequence of dates
- `DatetimeIndex` — the index type that unlocks time-aware operations
- `.resample()` — group and aggregate by time frequency

**Read:** [Pandas Time Series Documentation](https://pandas.pydata.org/docs/user_guide/timeseries.html)

In [ ]:
# --- EXAMPLE: Creating a time series in pandas ---

# Generate a daily date range
dates = pd.date_range(start='2023-01-01', end='2023-12-31', freq='D')

# Create a simple series with random values
np.random.seed(42)
values = np.random.randn(len(dates)).cumsum() + 100

ts = pd.Series(values, index=dates, name='price')
print("Type of index:", type(ts.index))
print("\nFirst 5 entries:")
print(ts.head())
print("\nSlicing by date string (January only):")
print(ts['2023-01'].head())

### Exercise 1.1 — Build Your Own Time Series

Using `pd.date_range`, create a **monthly** time series from `2020-01` to `2024-12` representing simulated monthly sales figures (use `np.random.randint(500, 2000, ...)` for values). Store it in a variable called `sales`.

In [ ]:
# YOUR CODE HERE
np.random.seed(0)

# 1. Create a monthly date range from 2020-01 to 2024-12
dates_monthly = ...

# 2. Create random integer sales values
sales_values = ...

# 3. Create a pd.Series named 'sales'
sales = ...

print(sales)

### Exercise 1.2 — Resampling

Resample `sales` to **quarterly** frequency using `.resample('QE').sum()` and store the result in `sales_quarterly`. Print both the original and resampled series length.

In [ ]:
# YOUR CODE HERE
sales_quarterly = ...

print("Monthly length:", ...)
print("Quarterly length:", ...)
print(sales_quarterly)

---
## Section 2 — Visualization with Matplotlib

Good time series plots communicate trends and patterns clearly. Key practices:
- Always label axes with units and time period
- Use `fig, ax = plt.subplots()` for fine-grained control
- Format the x-axis with `mdates` for readable date ticks
- Use `tight_layout()` to prevent label clipping

**Read:** [Matplotlib Time Series Plotting](https://matplotlib.org/stable/gallery/text_labels_and_annotations/date.html)

In [ ]:
# --- EXAMPLE: Proper time series plot ---
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(ts.index, ts.values, color='steelblue', linewidth=1.5, label='Price')
ax.set_title('Example Daily Price Series (2023)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Price')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 2.1 — Plot Your Sales Series

Plot your `sales` series (monthly) and `sales_quarterly` on the **same figure** with two subplots (one above the other). Include titles, axis labels, and formatted x-axis ticks.

In [ ]:
# YOUR CODE HERE
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Top subplot: monthly sales
# ...

# Bottom subplot: quarterly sales (use bar chart — ax2.bar(...))
# ...

plt.tight_layout()
plt.show()

---
## Section 3 — Time Series Components

Any time series can be decomposed into four components:

| Component | Description | Example |
|-----------|-------------|---------|
| **Trend** | Long-run direction (up/down/flat) | Revenue growing year-over-year |
| **Seasonality** | Regular, repeating patterns at fixed intervals | Higher retail sales every December |
| **Cyclic variation** | Wave-like fluctuations over non-fixed periods | Business/economic cycles |
| **Random / Residual** | Irregular, unpredictable noise | One-off supply chain disruption |

The **additive model** assumes: `y(t) = Trend + Seasonality + Cyclic + Random`  
The **multiplicative model** assumes: `y(t) = Trend × Seasonality × Cyclic × Random`  
Use multiplicative when variance grows with the level of the series.

In [ ]:
# --- EXAMPLE: Synthetic series with known components ---

np.random.seed(7)
t = np.arange(120)  # 10 years of monthly data
dates_ex = pd.date_range('2014-01', periods=120, freq='ME')

trend      = 0.5 * t                              # upward linear trend
seasonality = 15 * np.sin(2 * np.pi * t / 12)    # annual seasonal cycle
cyclic     = 8 * np.sin(2 * np.pi * t / 48)      # ~4-year business cycle
random     = np.random.normal(0, 3, 120)          # random noise

y = trend + seasonality + cyclic + random
series_ex = pd.Series(y, index=dates_ex, name='synthetic')

fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=True)
for ax, data, title, color in zip(
    axes,
    [y, trend, seasonality, cyclic, random],
    ['Full Series', 'Trend', 'Seasonality', 'Cyclic', 'Random'],
    ['black', 'royalblue', 'darkorange', 'green', 'crimson']
):
    ax.plot(dates_ex, data, color=color, linewidth=1.2)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Value')

plt.tight_layout()
plt.show()

### Exercise 3.1 — Seasonal Decomposition

Use `seasonal_decompose` from `statsmodels` to decompose `series_ex`. Set `model='additive'` and `period=12`. Plot all four components using the result's `.plot()` method.

In [ ]:
# YOUR CODE HERE
decomposition = ...

fig = decomposition.plot()
fig.set_size_inches(14, 10)
plt.tight_layout()
plt.show()

### Exercise 3.2 — Short Answer

Answer the following in the markdown cell below:

1. What does the **residual** component from the decomposition tell you? What would a large residual suggest about a data point?
2. When would you prefer a **multiplicative** decomposition over an additive one? Give a real-world example.

**Your answers here:**

1. 

2. 

---
## Section 4 — Moving Averages

A **moving average** (rolling mean) smooths out short-term fluctuations to reveal underlying trends. In pandas:

```python
series.rolling(window=7).mean()   # 7-period simple moving average
series.ewm(span=7).mean()         # exponentially weighted moving average
```

Key parameters:
- `window` — number of periods to average
- `min_periods` — minimum observations needed (default = window)
- `center=True` — center the window instead of trailing

**Read:** [Moving Averages in Pandas](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html)

In [ ]:
# --- EXAMPLE: SMA vs EMA ---

sma_7  = series_ex.rolling(window=7).mean()
sma_12 = series_ex.rolling(window=12).mean()
ema_12 = series_ex.ewm(span=12, adjust=False).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(series_ex.index, series_ex, alpha=0.4, color='gray', label='Original')
ax.plot(sma_7.index,  sma_7,  color='royalblue',   linewidth=2, label='SMA-7')
ax.plot(sma_12.index, sma_12, color='darkorange',   linewidth=2, label='SMA-12')
ax.plot(ema_12.index, ema_12, color='crimson',      linewidth=2, label='EMA-12', linestyle='--')
ax.set_title('Simple vs Exponential Moving Averages')
ax.set_xlabel('Date')
ax.set_ylabel('Value')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 4.1 — Rolling Statistics on Sales

On your `sales` series, compute:
- A **3-month SMA**
- A **12-month SMA** (annual trend)
- A **rolling standard deviation** with window=6

Plot the original series alongside both SMAs on one axis, and the rolling std on a second axis below.

In [ ]:
# YOUR CODE HERE
sma_3  = ...
sma_12 = ...
roll_std = ...

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: original + SMAs
# ...

# Bottom: rolling std
# ...

plt.tight_layout()
plt.show()

### Exercise 4.2 — Short Answer

1. Why does the SMA produce `NaN` values at the start of the series? How could `min_periods` help?
2. What is the main advantage of EMA over SMA for forecasting recent trends?

**Your answers here:**

1. 

2. 

---
## Section 5 — Lag Features

A **lag feature** is the value of a series at a previous time step. They are the most important input features for time series forecasting models because they encode the *memory* of the series.

```python
df['lag_1'] = df['value'].shift(1)   # value from 1 step ago
df['lag_7'] = df['value'].shift(7)   # value from 7 steps ago
```

**Why they matter:** A model predicting tomorrow's value benefits from knowing yesterday's value (`lag_1`), last week's value (`lag_7`), and last year's same-week value (`lag_52` for weekly data). In a multi-agent forecasting system, different agents may specialize in different lag horizons.

**Read:** [Lag Features Explained](https://machinelearningmastery.com/basic-feature-engineering-time-series-data-python/)

In [ ]:
# --- EXAMPLE: Building a lag feature DataFrame ---

df_lags = pd.DataFrame({'value': series_ex})

for lag in [1, 2, 3, 6, 12]:
    df_lags[f'lag_{lag}'] = df_lags['value'].shift(lag)

print("Shape:", df_lags.shape)
print("\nFirst 15 rows (notice NaN at start):")
print(df_lags.head(15).round(2))

In [ ]:
# --- EXAMPLE: Lag correlation plot ---
# How correlated is each lag with the current value?

df_clean = df_lags.dropna()
correlations = {col: df_clean['value'].corr(df_clean[col]) 
                for col in df_clean.columns if col != 'value'}

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(correlations.keys(), correlations.values(), color='steelblue', edgecolor='black')
ax.set_title('Correlation of Lag Features with Current Value')
ax.set_xlabel('Lag Feature')
ax.set_ylabel('Pearson Correlation')
ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

### Exercise 5.1 — Lag Features for Sales

Build a DataFrame from your `sales` series with lag features for lags: 1, 2, 3, 6, and 12 months. Then:
1. Drop rows with `NaN` values
2. Compute and print a **correlation matrix** using `.corr()`
3. Plot the correlation of each lag feature with the target (`sales`) as a bar chart

In [ ]:
# YOUR CODE HERE
df_sales = pd.DataFrame({'sales': sales})

# 1. Add lag features
for lag in [1, 2, 3, 6, 12]:
    df_sales[f'lag_{lag}'] = ...

# 2. Drop NaNs
df_sales = ...

# 3. Correlation matrix
print(df_sales.corr().round(3))

# 4. Bar chart of lag correlations with target
# ...

### Exercise 5.2 — Short Answer

1. In the context of multi-agent forecasting: if one agent uses `lag_1` and `lag_2` while another uses `lag_12`, what different *aspects* of the series is each agent likely capturing?
2. What problem arises if you use a lag feature without dropping `NaN` rows before training a model?

**Your answers here:**

1. 

2. 

---
## Section 6 — Putting It All Together

This final section combines everything: load a real-world style dataset, visualize it, decompose it, and engineer features from it.

We'll use the **Air Passengers** dataset — monthly international airline passengers from 1949–1960. It's a classic benchmark that exhibits clear trend, seasonality, and multiplicative growth.

In [ ]:
# Load the Air Passengers dataset
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
air = pd.read_csv(url, parse_dates=['Month'], index_col='Month')
air.columns = ['passengers']
air.index.freq = 'MS'  # month start
print(air.head())
print("\nShape:", air.shape)

### Exercise 6.1 — Full EDA

Perform a complete exploratory analysis of the `air` dataset:

1. Plot the raw series
2. Overlay a 12-month SMA
3. Run a **multiplicative** seasonal decomposition (`model='multiplicative'`, `period=12`) and plot it
4. Build lag features for lags 1, 12 (same month last year), and 24
5. Plot the lag-12 correlation scatter plot: `air['passengers']` (y-axis) vs `lag_12` (x-axis). What does this tell you?

In [ ]:
# Part 1 & 2: Raw series + 12-month SMA
# YOUR CODE HERE


In [ ]:
# Part 3: Multiplicative decomposition
# YOUR CODE HERE


In [ ]:
# Part 4 & 5: Lag features + scatter plot
# YOUR CODE HERE


### Exercise 6.2 — Reflection

Answer in 3–5 sentences: Based on your analysis of the Air Passengers dataset, which components (trend, seasonality, cyclic, random) are most prominent? Which lag features would you prioritize as inputs to a forecasting model and why? How does this connect to the idea of having specialized agents in a multi-agent forecasting system?

**Your answer here:**



---
## Bonus Challenge ⭐

Combine moving averages and lag features into a single feature-engineered DataFrame for the Air Passengers dataset with the following columns:
- `passengers` (target)
- `lag_1`, `lag_12`
- `sma_3`, `sma_12`
- `rolling_std_6`
- `month` (integer 1–12, extracted from the index) — this encodes seasonality

Drop NaN rows and print the final shape and first 5 rows. This DataFrame is what you'd feed into a forecasting model in Week 2.

In [ ]:
# BONUS — YOUR CODE HERE


---
## Submission Checklist

Before submitting, make sure:
- [ ] All `# YOUR CODE HERE` cells are filled in and run without errors
- [ ] All short-answer cells are filled in
- [ ] All plots have titles, axis labels, and legends
- [ ] The notebook runs top-to-bottom with **Kernel → Restart & Run All**
- [ ] Your name and date are filled in at the top

**Submit as:** `Week1_Assignment_YourName.ipynb`